In [ ]:
import json
import pandas as pd

with open("PPC_Urdu.json","r",encoding="utf-8") as f:
    ppc = json.load(f)

ppc_df = pd.DataFrame(ppc)

ppc_df.head()

,section,title,description,text
0,1,ضابطے کا نام اور نفاذ کی حدود,اس قانون کو پاکستان پینل کوڈ کہا جائے گا اور ی...,1 ضابطے کا نام اور نفاذ کی حدود اس قانون کو پا...
1,2,پاکستان کے اندر کیے گئے جرائم کی سزا,پاکستان کے اندر جو بھی شخص اس قانون کے خلاف کو...,2 پاکستان کے اندر کیے گئے جرائم کی سزا پاکستان...
2,3,پاکستان سے باہر کیے گئے مگر پاکستان میں قابلِ ...,جو شخص پاکستان کے قانون کے مطابق پاکستان سے با...,3 پاکستان سے باہر کیے گئے مگر پاکستان میں قابل...
3,4,پاکستان سے باہر کیے گئے جرائم پر ضابطے کا اطلاق,اس قانون کی دفعات ان جرائم پر بھی لاگو ہوں گی ...,4 پاکستان سے باہر کیے گئے جرائم پر ضابطے کا اط...
4,5,ایسے قوانین جن پر اس قانون کا اثر نہیں ہوگا,اس قانون کا مقصد ریاستی افواج افسران سپاہیوں م...,5 ایسے قوانین جن پر اس قانون کا اثر نہیں ہوگا ...


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

documents = []

for _, row in ppc_df.iterrows():
    documents.append(str(row["text"]).strip())
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-m3")

documents = []

for _, row in ppc_df.iterrows():
    documents.append(str(row["text"]).strip())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [ ]:
embeddings = model.encode(
    documents,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
!pip install -q faiss-cpu
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype("float32"))

print("Total PPC Sections:", index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.5 MB/s eta 0:00:00
Total PPC Sections: 577


In [ ]:
fir_df = pd.read_csv(
    "FIR_Test.csv"
)

fir_df.head()
print(fir_df.columns.tolist())

['Text', 'Trigger_words ', 'Applied_Sections', 'Crime_category']


In [ ]:
fir_df = pd.read_csv(
    "Original_FIR.csv"
)

fir_df.head()
print(fir_df.columns.tolist())

['FIR_ID', 'Text', 'Trigger_words ', 'Applied_Sections', 'Crime_category', 'Normalized_Category', 'Unnamed: 6']


In [ ]:
retrieved_sections = []

for _, row in fir_df.iterrows():

    query = f"""
    Crime: {row['Crime_category']}
    Keywords: {row['Trigger_words ']}
    FIR: {row['Text']}
    """

    q = model.encode(
        "query: " + query,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    D, I = index.search(
        np.array([q], dtype=np.float32),
        k=3
    )

    top3 = [str(ppc_df.iloc[idx]["section"]) for idx in I[0]]

    retrieved_sections.append(" / ".join(top3))

fir_df["PPC_Retrieved_Sections"] = retrieved_sections

In [ ]:
fir_df["PPC_Retrieved_Sections"] = retrieved_sections

In [ ]:
import pandas as pd
import numpy as np

results = []

for _, row in fir_df.iterrows():

    query = f"""
    Crime: {row['Crime_category']}
    Keywords: {row['Trigger_words ']}
    FIR: {row['Text']}
    """

    # Query embedding
    q = model.encode(
        "query: " + query,
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    # Top-3 PPC retrieval
    D, I = index.search(
        np.array([q], dtype=np.float32),
        k=3
    )

    # Retrieved PPC sections
    top_sections = [
        str(ppc_df.iloc[idx]["section"])
        for idx in I[0]
    ]

    results.append({
        "FIR Query": row["Text"],
        "Crime Category": row["Crime_category"],
        "Trigger Words": row["Trigger_words "],
        "Original Applied Sections": row["Applied_Sections"],
        "Top-1 PPC": top_sections[0],
        "Top-2 PPC": top_sections[1],
        "Top-3 PPC": top_sections[2]
    })


# DataFrame
result_df = pd.DataFrame(results)

# Case No
result_df.insert(
    0,
    "Case No",
    range(1, len(result_df) + 1)
)

# Save
save_path = "/content/drive/MyDrive/FIR_PROJECT/PPC_200_Top3_Retrieval_Results.xlsx"

result_df.to_excel(
    save_path,
    index=False
)

print("=" * 60)
print("PPC Top-3 Retrieval completed successfully!")
print(f"Total FIRs: {len(result_df)}")
print(f"Saved: {save_path}")
print("=" * 60)

display(result_df.head(10))

PPC Top-3 Retrieval completed successfully!
Total FIRs: 200
Saved: /content/drive/MyDrive/FIR_PROJECT/PPC_200_Top3_Retrieval_Results.xlsx


,Case No,FIR Query,Crime Category,Trigger Words,Original Applied Sections,Top-1 PPC,Top-2 PPC,Top-3 PPC
0,1,[مدعی] مندرجہ خانہ نمبر 2 نے بحاضری تھانہ آکر ...,جنسی جرائم + دھمکی,زنا، بھائی، قتل کی دھمکی، چاقو,"376, 506B",509,375,496-ب
1,2,استغاثہ زیر دفعہ 285/286 ت پ اس وقت مظہر [اہلک...,عوامی تحفظ / خطرناک مواد,LPG سلنڈر، غیر قانونی ایندھن، ناکہ بندی,285/286,171,481,263-الف
2,3,بمطابق بیان زیر دفعہ 154 ض ف [مدعی] ساکن رفاء ...,ڈکیتی / رہزنی,گھر میں ڈکیتی، مسلح ملزمان، سونا و نقدی,392/397/34,390,410,457
3,4,زبانی [مدعی] میں خانہ نمبر 2 اپنے اہل و عیال ک...,قتل / اقدامِ قتل,فائرنگ، ادھار رقم، زخمی، اقدامِ قتل,324/34,103,171,337-ح
4,5,زبانی [مدعی] پتہ بالا خانہ نمبر 2 میں درج اڈری...,ڈکیتی / رہزنی,بینک سے تعاقب، اسلحہ کے زور پر، چھینا جھپٹی,392/397/34,390,171,489-ٹ
5,6,موصولہ بیان زیر دفعہ 154 ض ف لفظ به لفظ ذیل ہے...,ڈکیتی / رہزنی,سپلائر، شیشہ توڑنا، کیش چھیننا,382/427/34,171,390,489-ٹ
6,7,زبانی [مدعی] [مدعی پتہ] میں رہائش پزیر ہوں [پر...,چوری + جائیداد میں مداخلت,کاپر وائر، سب اسٹیشن، چوری,380/454,410,381,390
7,8,بجے من [اہلکار] بحیثیت ڈیوٹی آفیسر کمرہ رپورٹن...,مالیاتی دھوکہ دہی,انویسٹمنٹ فراڈ، باؤنس چیک، کروڑوں کی رقم,489F,409,171,420
8,9,زبانی [مدعی] میں [مدعی پتہ] پر اپنی فیملی کے س...,اغوا,بچی کا اغوا، لاپتہ، پڑوسی پر شک,364,369,366-الف,361
9,10,زبانی [مدعی] میں [مدعی پتہ] معہ اپنی فیملی کے ...,ڈکیتی / رہزنی,موٹر سائیکل چھیننا، اسلحہ، آفس جاتے ہوئے,392/397/34,390,381-الف,171


In [ ]:
import pandas as pd

# ===========================
# Load Retrieval File
# ===========================
input_file = "/content/drive/MyDrive/FIR_PROJECT/PPC_Top3_Retrieval_Results.xlsx"

df = pd.read_excel(input_file)

# ===========================
# Helper Function
# ===========================
def split_sections(x):
    if pd.isna(x):
        return set()

    x = str(x).replace(",", "/")

    return set(i.strip() for i in x.split("/") if i.strip())

# ===========================
# Top-3 Jaccard
# ===========================
scores = []

for _, row in df.iterrows():

    # Ground Truth
    true_set = split_sections(row["Original Applied Sections"])

    # Predicted = Top1 + Top2 + Top3
    pred_set = set()

    pred_set |= split_sections(row["Top-1 PPC"])
    pred_set |= split_sections(row["Top-2 PPC"])
    pred_set |= split_sections(row["Top-3 PPC"])

    union = true_set | pred_set

    if len(union) == 0:
        score = 1.0
    else:
        score = len(true_set & pred_set) / len(union)

    scores.append(score)

# Save scores
df["PPC_Top3_Jaccard"] = scores

# Average
avg_jaccard = sum(scores) / len(scores)

print("=" * 60)
print(f"Average PPC Top-3 Jaccard : {avg_jaccard:.4f}")
print("=" * 60)

# Save
output_file = "/content/drive/MyDrive/FIR_PROJECT/PPC_Top3_Jaccard_Results.xlsx"

df.to_excel(output_file, index=False)

print("Saved:", output_file)

Average PPC Top-3 Jaccard : 0.0862
Saved: /content/drive/MyDrive/FIR_PROJECT/PPC_Top3_Jaccard_Results.xlsx


In [3]:
import pandas as pd

# ==========================================================
# Load Retrieval File
# ==========================================================
input_file = "/content/drive/MyDrive/FIR_PROJECT/PPC_Top3_Retrieval_Results.xlsx"

df = pd.read_excel(input_file)

# ==========================================================
# Helper Function
# ==========================================================
def split_sections(x):
    if pd.isna(x):
        return set()

    x = str(x).replace(",", "/")

    return set(
        i.strip().upper()
        for i in x.split("/")
        if i.strip()
    )

# ==========================================================
# Helper: Jaccard Similarity
# ==========================================================
def jaccard_score(true_set, pred_set):

    union = true_set | pred_set

    if len(union) == 0:
        return 1.0

    return len(true_set & pred_set) / len(union)

# ==========================================================
# Store Results
# ==========================================================
top1_jaccard = []
top2_jaccard = []
top3_jaccard = []

top1_hamming = []
top2_hamming = []
top3_hamming = []

top3_hits = []

# ==========================================================
# Create Label Universe for Hamming Loss
# ==========================================================
all_sections = set()

for _, row in df.iterrows():

    all_sections |= split_sections(
        row["Original Applied Sections"]
    )

    all_sections |= split_sections(
        row["Top-1 PPC"]
    )

    all_sections |= split_sections(
        row["Top-2 PPC"]
    )

    all_sections |= split_sections(
        row["Top-3 PPC"]
    )

all_sections = sorted(all_sections)

print("Total PPC Sections in Evaluation:", len(all_sections))

# ==========================================================
# Main Evaluation
# ==========================================================
for _, row in df.iterrows():

    # --------------------------
    # Ground Truth
    # --------------------------
    true_set = split_sections(
        row["Original Applied Sections"]
    )

    # --------------------------
    # Top-1
    # --------------------------
    top1_set = split_sections(
        row["Top-1 PPC"]
    )

    # --------------------------
    # Top-2
    # --------------------------
    top2_set = split_sections(
        row["Top-2 PPC"]
    )

    # --------------------------
    # Top-3
    # --------------------------
    top3_set = split_sections(
        row["Top-3 PPC"]
    )

    # ======================================================
    # Jaccard
    # ======================================================

    j1 = jaccard_score(true_set, top1_set)

    j2 = jaccard_score(
        true_set,
        top1_set | top2_set
    )

    j3 = jaccard_score(
        true_set,
        top1_set | top2_set | top3_set
    )

    top1_jaccard.append(j1)
    top2_jaccard.append(j2)
    top3_jaccard.append(j3)

    # ======================================================
    # Hamming Loss
    # ======================================================

    true_vector = [
        1 if section in true_set else 0
        for section in all_sections
    ]

    pred1_vector = [
        1 if section in top1_set else 0
        for section in all_sections
    ]

    pred2_vector = [
        1 if section in (top1_set | top2_set) else 0
        for section in all_sections
    ]

    pred3_vector = [
        1 if section in (top1_set | top2_set | top3_set) else 0
        for section in all_sections
    ]

    h1 = sum(
        a != b
        for a, b in zip(true_vector, pred1_vector)
    ) / len(all_sections)

    h2 = sum(
        a != b
        for a, b in zip(true_vector, pred2_vector)
    ) / len(all_sections)

    h3 = sum(
        a != b
        for a, b in zip(true_vector, pred3_vector)
    ) / len(all_sections)

    top1_hamming.append(h1)
    top2_hamming.append(h2)
    top3_hamming.append(h3)

    # ======================================================
    # Top-3 Retrieval Hit
    # At least one correct PPC section retrieved
    # ======================================================

    hit = 1 if len(true_set & (
        top1_set | top2_set | top3_set
    )) > 0 else 0

    top3_hits.append(hit)

# ==========================================================
# Add Results to DataFrame
# ==========================================================

df["Top-1 Jaccard"] = top1_jaccard
df["Top-2 Jaccard"] = top2_jaccard
df["Top-3 Jaccard"] = top3_jaccard

df["Top-1 Hamming Loss"] = top1_hamming
df["Top-2 Hamming Loss"] = top2_hamming
df["Top-3 Hamming Loss"] = top3_hamming

df["Top-3 Hit"] = top3_hits

# ==========================================================
# Overall Metrics
# ==========================================================

avg_top1_jaccard = sum(top1_jaccard) / len(top1_jaccard)
avg_top2_jaccard = sum(top2_jaccard) / len(top2_jaccard)
avg_top3_jaccard = sum(top3_jaccard) / len(top3_jaccard)

avg_top1_hamming = sum(top1_hamming) / len(top1_hamming)
avg_top2_hamming = sum(top2_hamming) / len(top2_hamming)
avg_top3_hamming = sum(top3_hamming) / len(top3_hamming)

top3_accuracy = sum(top3_hits) / len(top3_hits)

# Best Hamming per FIR
best_hamming_per_case = [
    min(h1, h2, h3)
    for h1, h2, h3 in zip(
        top1_hamming,
        top2_hamming,
        top3_hamming
    )
]

best_hamming = (
    sum(best_hamming_per_case)
    / len(best_hamming_per_case)
)

# ==========================================================
# Print Results
# ==========================================================

print("=" * 60)
print("PPC TOP-3 RETRIEVAL PERFORMANCE")
print("=" * 60)

print(f"Top-1 Jaccard        : {avg_top1_jaccard:.4f} "
      f"({avg_top1_jaccard*100:.2f}%)")

print(f"Top-2 Jaccard        : {avg_top2_jaccard:.4f} "
      f"({avg_top2_jaccard*100:.2f}%)")

print(f"Top-3 Jaccard        : {avg_top3_jaccard:.4f} "
      f"({avg_top3_jaccard*100:.2f}%)")

print("-" * 60)

print(f"Top-1 Hamming Loss   : {avg_top1_hamming:.4f}")
print(f"Top-2 Hamming Loss   : {avg_top2_hamming:.4f}")
print(f"Top-3 Hamming Loss   : {avg_top3_hamming:.4f}")

print(f"Best Hamming Loss    : {best_hamming:.4f}")

print("-" * 60)
# ==========================================================
# Overall Average Jaccard & Hamming
# ==========================================================

avg_jaccard = (
    avg_top1_jaccard +
    avg_top2_jaccard +
    avg_top3_jaccard
) / 3

avg_hamming = (
    avg_top1_hamming +
    avg_top2_hamming +
    avg_top3_hamming
) / 3

print(f"Average Jaccard Similarity : {avg_jaccard:.4f} "
      f"({avg_jaccard*100:.2f}%)")

print(f"Average Hamming Loss       : {avg_hamming:.4f}")

print(f"Top-3 Retrieval Accuracy : {top3_accuracy:.4f} "
      f"({top3_accuracy*100:.2f}%)")

print("=" * 60)

# ==========================================================
# Save Results
# ==========================================================

output_file = (
    "/content/drive/MyDrive/FIR_PROJECT/"
    "PPC_Top3_Jaccard_Hamming_Results.xlsx"
)

df.to_excel(output_file, index=False)

print("\nSaved:", output_file)

Total PPC Sections in Evaluation: 104
PPC TOP-3 RETRIEVAL PERFORMANCE
Top-1 Jaccard        : 0.1633 (16.33%)
Top-2 Jaccard        : 0.1093 (10.93%)
Top-3 Jaccard        : 0.0862 (8.62%)
------------------------------------------------------------
Top-1 Hamming Loss   : 0.0226
Top-2 Hamming Loss   : 0.0312
Top-3 Hamming Loss   : 0.0400
Best Hamming Loss    : 0.0221
------------------------------------------------------------
Average Jaccard Similarity : 0.1196 (11.96%)
Average Hamming Loss       : 0.0312
Top-3 Retrieval Accuracy : 0.2933 (29.33%)

Saved: /content/drive/MyDrive/FIR_PROJECT/PPC_Top3_Jaccard_Hamming_Results.xlsx
